# Jour 5: Passons au niveau PRO !
### Technique avances de RAG

#### Commençons par approfondir l'étape d'ingestion :

1. Pas de LangChain ! Uniquement du natif pour une flexibilité maximale.

2. Utilisons un LLM pour diviser les segments (chunks) de manière intelligente.

3. Utilisons la meilleure taille de segment et le meilleur encodeur (embedding) identifiés hier.

4. Demandons également au LLM de réécrire les segments de la manière la plus utile possible (« pré-traitement de documents »).  

In [51]:
import os
import json
import numpy as np
import plotly.graph_objects as go

from sentence_transformers import SentenceTransformer
from groq import Groq
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from pathlib import Path
from litellm import completion
from sklearn.manifold import TSNE
from pathlib import Path

# Chargement des variables d'environnement
load_dotenv(override=True)

# Configuration du modèle (Note : gpt-4.1-nano est une version optimisée pour la vitesse)
MODEL_3 = "groq/gpt-oss-120B"
MODEL_2 = "groq/llama-3.3-70b-versatile"
MODEL = "google/gemma-3-27b-it:free"

# Paramètres de la base de données et du dossier de connaissances
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "All-MiniLM-L6-v2"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500
RETRIEVAL_K = 5

In [26]:
# 1. On met la clé directement en chaîne de caractères (SANS passer par os.getenv)
# Assure-toi qu'il n'y a pas d'espace avant le 'sk-or-v1' ni à la fin.
MA_CLE_TEST = os.getenv("OPENROUTER_API_KEY") 

# 2. On initialise un client de test pur
test_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=MA_CLE_TEST
)

# 3. On fait un appel standard (pas de parse, juste pour tester la connexion)
try:
    print("Tentative de connexion...")
    response = test_client.chat.completions.create(
        model="google/gemma-3-27b-it:free",
        messages=[{"role": "user", "content": "Réponds juste par 'Pong'."}]
    )
    print("✅ SUCCÈS :", response.choices[0].message.content)
except Exception as e:
    print("❌ ECHEC :", e)

Tentative de connexion...
❌ ECHEC : Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-3-27b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3BxN7GixBm7cwAuCnO0Gd7K21Kj'}


In [53]:
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set in the environment variables.")


client = OpenAI(
    base_url="https://groq.com/api/v1",
    api_key=groq_api_key
)

# On fait un appel standard (pas de parse, juste pour tester la connexion)
try:
    print("Tentative de connexion...")
    response = test_client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": "Réponds juste par 'Pong'."}]
    )
    print("✅ SUCCÈS :", response.choices[0].message.content)
except Exception as e:
    print("❌ ECHEC :", e)

Tentative de connexion...
✅ SUCCÈS : Pong


In [2]:
# Utilisation du model local 
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

MODEL = "llama3.2:3b"

In [ ]:
from pydantic import BaseModel
from typing import List



In [64]:
# Inspirared by Langchain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [66]:
# Class ot perfecfly represent a chunk
class Chunk(BaseModel):
    headline: str = Field(description="  brief heading for this chunk, typically a few words, that is most likely to be surfaced in a search query")
    summary: str = Field(description="A feq sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of the chunk, which may be truncated to fit within the average chunk size")
    
    def as_result(self, document):
        page_content = self.headline + "\n\n" + self.summary + "\n\n" + self.original_text
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=page_content, metadata=metadata)

class Chunks(BaseModel):
    chunks: list[Chunk]

## Trois etapes
1. Récupérer les documents de la base de connaissances (Knowledge Base), comme le faisait LangChain.

2. Appeler un LLM pour transformer les documents en segments (Chunks).

3. Stocker les segments dans Chroma.

C'est tout !

### Commençons par l'Étape 1

In [67]:
def fetch_documents():
    """Une version faite maison du DirectoryLoader de LangChain"""
    documents = []
    
    # Parcourt chaque dossier dans le chemin de la base de connaissances
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        # Cherche tous les fichiers Markdown (*.md) de manière récursive
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({
                    "type": doc_type, 
                    "source": file.as_posix(), 
                    "text": f.read()
                })
                
    print(f"Chargé {len(documents)} documents")
    return documents

In [68]:
documents = fetch_documents()

Chargé 32 documents


### Etape 2 - faire les segments ou morceaux

In [70]:
def make_prompt(document):
    """Génère le prompt pour demander au LLM de découper le document"""
    # Calcul du nombre approximatif de segments souhaités
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text...

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:
"""

In [82]:
MODEL = "groq/meta-llama/llama-4-scout-17b-16e-instruct"

In [ ]:
MODEL = "groq/meta-llama/llama-3.1-8b-instant"

In [84]:
def process_document(doc):
    prompt = make_prompt(doc)
    
    # On utilise litellm (ou groq directement) pour obtenir une réponse structurée
    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=Chunks # On utilise ta classe Pydantic définie plus haut
    )
    
    # On récupère le contenu et on le transforme en liste de Chunks
    raw_reply = response.choices[0].message.content
    structured_chunks = Chunks.model_validate_json(raw_reply)
    
    # Conversion en format 'Result' pour être compatible avec la suite du code
    return [c.as_result(doc) for c in structured_chunks.chunks]  

### Test

In [60]:
def process_document(doc):
    """Envoie le document au LLM et convertit le résultat au bon format"""
    prompt = make_prompt(doc)
    
    # Appel au LLM avec le SDK OpenAI
    response = client.beta.chat.completions.parse(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {"role": "system", "content": "Tu es un expert en découpage sémantique de documents."},
            {"role": "user", "content": prompt}
        ],
        response_format=Chunks # Ta classe Pydantic
    )
    
    # On récupère les objets 'Chunk' générés
    parsed_chunks = response.choices[0].message.parsed.chunks
    
    # CORRECTION : On les convertit en objets 'Result' pour avoir "page_content"
    return [c.as_result(doc) for c in parsed_chunks]

In [75]:
# Initialisation du client Groq
client_groq = Groq(
    api_key=os.getenv("GROQ_API_KEY"),
)

In [79]:
def process_document(doc):
    """Version universelle compatible OpenRouter / Groq / OpenAI"""
    prompt = make_prompt(doc)
    
    # 1. Utiliser .create() au lieu de .beta...parse()
    # On ajoute response_format={"type": "json_object"}
    response = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant", # Rapide et économique pour du chunking
        messages=[
            {
                "role": "system", 
                "content": "Tu es un expert en RAG. Tu dois extraire les segments de texte et répondre UNIQUEMENT avec un objet JSON valide."
            },
            {"role": "user", "content": prompt}
        ],
            # Activation du mode JSON
        response_format={"type": "json_object"}
    )
    
# 1. Parsing du JSON brut vers l'objet Pydantic 'Chunks'
    raw_json = response.choices[0].message.content
    parsed_data = Chunks.model_validate(raw_json)
    
# 2. Conversion magique de chaque 'Chunk' en 'Result'
    return [c.as_result(doc) for c in parsed_data.chunks]

In [76]:
def process_document(doc):
    prompt = make_prompt(doc)
    
    try:
        # On utilise le client Groq ici
        response = client_groq.chat.completions.create(
            model="llama-3.1-8b-instant", # Rapide et économique pour du chunking
            messages=[
                {
                    "role": "system", 
                    "content": "Tu es un expert en RAG. Tu dois extraire les segments de texte et répondre UNIQUEMENT avec un objet JSON valide."
                },
                {"role": "user", "content": prompt}
            ],
            # Activation du mode JSON
            response_format={"type": "json_object"}
        )
        
        # Extraction et parsing
        raw_content = response.choices[0].message.content
        data_json = json.loads(raw_content)
        
        # Validation Pydantic (on garde notre structure "Pro")
        parsed_data = Chunks.model_validate(data_json)
        
        # Conversion en objets 'Result' (pour avoir page_content et metadata)
        return [c.as_result(doc) for c in parsed_data.chunks]
        
    except Exception as e:
        print(f"❌ Erreur Groq sur le document {doc.metadata.get('source', 'inconnu')}: {e}")
        return []

### Fin test

In [85]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [86]:
Chunks = create_chunks(documents)

100%|██████████| 32/32 [07:16<00:00, 13.63s/it]


In [87]:
print(f"Created {len(Chunks)} chunks from {len(documents)} documents")

Created 197 chunks from 32 documents


## Etape 3 - sauvegarder sous formes de vecteurs

In [88]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [89]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)
        
    texts = [chunk.page_content for chunk in chunks]
    
    # MODIFICATION : Encodage local avec Hugging Face
    vectors = embedder.encode(texts).tolist() 
    
    collection = chroma.get_or_create_collection(name=collection_name)
    
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]
    
    collection.add(ids=ids, embeddings=vectors, metadatas=metas, documents=texts)
    print(f"Vectorstore créé avec {collection.count()} documents")

In [90]:
create_embeddings(Chunks)

Vectorstore créé avec 197 documents


## Ne peut on pas faire plus.. vraiment?

### Visualisation

In [93]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_collection(name=collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
color_map = {
    'products': 'blue',
    'employees': 'red',
    'contacts': 'green',
    'company': 'yellow'
}
colors = [color_map.get(t, 'grey') for t in doc_types]

#### En 2D

In [100]:
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
vectors_2d = tsne_2d.fit_transform(vectors)

# Create the 2D scarte
fig_2d = go.Figure(data=[go.Scatter(
    x=vectors_2d[:, 0],
    y=vectors_2d[:, 1],
    mode='markers',
    marker=dict(size=7, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Texte: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig_2d.update_layout(
    title="2D Vector Store Visualization (ChromaBD)", 
    xaxis_title='Axe X', 
    yaxis_title='Axe Y',
    width=800,
    height=600,
    margin=dict(l=0, r=0, b=0, t=30)
)

fig_2d.show()


#### En 3D

In [101]:
# T-SNE configuré pour 3 dimensions
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(30, len(vectors)-1))
vectors_3d = tsne_3d.fit_transform(vectors)

# Création du graphique 3D (go.Scatter3d au lieu de go.Scatter)
fig_3d = go.Figure(data=[go.Scatter3d(
    x=vectors_3d[:, 0],
    y=vectors_3d[:, 1],
    z=vectors_3d[:, 2], # Ajout vital de l'axe Z
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Texte: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig_3d.update_layout(
    title="3D Vector Store Visualization (ChromaBD)", 
    scene=dict(xaxis_title='Axe X', yaxis_title='Axe Y', zaxis_title='Axe Z'),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=30)
)

fig_3d.show()


## Et maintenant construisons un RAG avance

In [102]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [103]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve it.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question:\n\n"
    user_prompt += "Here are the chunks:\n\n"
    
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
        
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    # Appel au modèle via litellm (MODEL défini dans la capture 1)
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    
    # Validation du format JSON via Pydantic
    order = RankOrder.model_validate_json(reply).order
    print(f"Order of relevance (chunk ids): {order}")
    
    result = []
    for i in order:
        result.append(chunks[i - 1])
    # On retourne la liste des segments réordonnée
    return result

In [104]:
def fetch_context_unranked(question):
    # MODIFICATION : Encodage local de la question
    query_vector = embedder.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=RETRIEVAL_K)
    chunks = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        chunks.append(Result(page_content=doc, metadata=meta))
    return chunks

In [105]:
question = "Who is Jordan"
chunks = fetch_context_unranked(question)

In [106]:
for chunk in chunks:
    print(f"Chunk content: {chunk.page_content[:10]}")

Chunk content: Conclusion
Chunk content: Jordan Bla
Chunk content: Jordan Bla
Chunk content: Conclusion
Chunk content: Jordan Bla


In [107]:
question = "What are the different products offered by the company?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, chunk in enumerate(chunks):
    if "manchester" in chunk.page_content.lower():
        print(f"Chunk {index + 1} content: {chunk.page_content[:100]}...")

In [108]:
reranked = rerank(question, chunks) 

Order of relevance (chunk ids): [1, 2, 6, 9, 7, 3, 5, 4, 14, 8, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20]


In [110]:
for index, chunk in enumerate(reranked):
    if "manchester" in chunk.page_content.lower():
        print(f"Reranked Chunk {index + 1} content: {chunk.page_content[:100]}...")

In [111]:
reranked[0].page_content

'Product Offerings\n\nInsurellm offers a range of insurance products for individuals and businesses.\n\nWe offer a wide range of insurance products, including life insurance, health insurance, and property insurance. Our products are designed to provide financial protection and peace of mind.'

In [112]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)
    

In [113]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and actually answers the question.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [114]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [119]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refind question that you will use to search th knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role":"system", "content": message}])
    return response.choices[0].message.content

In [120]:
rewrite_query("Who is the IIOTY award?", [])

'What does IIOTY award stand for and who received it in relation to Insurellm?'

In [123]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [125]:
answer_question("Who is the IIOTY award?", [])

What does IIOTY award stand for and relation to Insurellm?
Order of relevance (chunk ids): [15, 11, 1, 17, 3, 18, 2, 4, 5, 12, 19, 6, 7, 8, 10, 13, 16, 20, 9, 14]


"I don't have information on the IIOTY award or its recipient. Can I help you with something else?"

## Interface Gradio

In [139]:
# Autre impoort
import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.manifold import TSNE
import chromadb

from evaluation.test import load_tests
from evaluation.eval import evaluate_retrieval, evaluate_answer

# ─── CONFIGURATION DE LA BASE VECTORIELLE ─────────────────────────────────────
# TODO: Modifie ces variables pour correspondre à ton projet
DB_PATH = "./preprocessed_db"
COLLECTION_NAME = "docs" 

# ─── CHARGEMENT DES TESTS (une fois) ──────────────────────────────────────────
tests = load_tests()
print(f"✅ {len(tests)} questions de test chargées")

✅ 152 questions de test chargées


In [140]:
# ─── LOGIQUE DE VISUALISATION DES VECTEURS ────────────────────────────────────

def get_vector_plot(dimension=2):
    """
    Récupère les vecteurs depuis ChromaDB, applique t-SNE et génère le Plotly.
    """
    try:
        # 1. Connexion et récupération des données
        client = chromadb.PersistentClient(path=DB_PATH)
        collection = client.get_collection(COLLECTION_NAME)
        data = collection.get(include=['embeddings', 'documents', 'metadatas'])
        
        vectors = np.array(data['embeddings'])
        documents = data['documents']
        metadatas = data['metadatas']
        
        if len(vectors) < 2:
            return go.Figure().update_layout(title="Pas assez de documents dans la base pour la visualisation.")

        # 2. Réduction de dimension avec t-SNE
        # La perplexité doit être inférieure au nombre d'échantillons
        perplexity = min(30, max(1, len(vectors) - 1)) 
        tsne = TSNE(n_components=dimension, random_state=42, perplexity=perplexity)
        reduced_vectors = tsne.fit_transform(vectors)
        
        # 3. Préparation du DataFrame pour Plotly
        df_viz = pd.DataFrame(
            reduced_vectors, 
            columns=[f'Dim_{i+1}' for i in range(dimension)]
        )
        
        # Sécurisation de l'extraction des métadonnées (évite les erreurs KeyError)
        df_viz['Type'] = [m.get('type', 'Inconnu') if m else 'Inconnu' for m in metadatas]
        df_viz['Source'] = [m.get('source', 'Inconnu') if m else 'Inconnu' for m in metadatas]
        # Limite le texte pour éviter d'alourdir l'affichage au survol
        df_viz['Texte'] = [doc[:150] + "..." if doc else "" for doc in documents]
        
        # 4. Génération de la figure Plotly Express
        if dimension == 2:
            fig = px.scatter(
                df_viz, x='Dim_1', y='Dim_2', 
                color='Type', 
                hover_data=['Source', 'Texte'], 
                title="Projection 2D des Segments de Texte (t-SNE)"
            )
        else:
            fig = px.scatter_3d(
                df_viz, x='Dim_1', y='Dim_2', z='Dim_3', 
                color='Type', 
                hover_data=['Source', 'Texte'], 
                title="Projection 3D de l'Espace Sémantique"
            )
            
        fig.update_traces(marker=dict(size=6, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
        fig.update_layout(template='plotly_white', margin=dict(l=0, r=0, b=0, t=40))
        return fig
        
    except Exception as e:
        fig = go.Figure()
        fig.update_layout(title=f"❌ Erreur lors du chargement des vecteurs : {str(e)}")
        return fig

def plot_2d_vectors(): return get_vector_plot(2)
def plot_3d_vectors(): return get_vector_plot(3)

In [141]:
# ─── ÉVALUATION RETRIEVAL ─────────────────────────────────────────────────────

def run_retrieval_eval():
    results = []
    for test in tests:
        res = evaluate_retrieval(test)
        results.append({
            "category": test.category,
            "question": test.question[:60] + "...",
            "mrr": res.mrr,
            "ndcg": res.ndcg,
            "keywords_coverage": res.keywords_coverage,
        })
    df = pd.DataFrame(results)
    
    mean_mrr = df["mrr"].mean()
    mean_ndcg = df["ndcg"].mean()
    mean_coverage = df["keywords_coverage"].mean()
    
    category_df = df.groupby("category", as_index=False)["mrr"].mean()
    
    fig = px.bar(
        category_df, x="category", y="mrr", title="Average MRR by Category",
        color="mrr", color_continuous_scale="RdYlGn",
        labels={"mrr": "Mean MRR", "category": "Category"},
    )
    fig.update_layout(yaxis_range=[0, 1], showlegend=False)
    
    def color_score(val):
        if val >= 0.75: return "🟢"
        elif val >= 0.55: return "🟡"
        else: return "🔴"
    
    stats_md = f"""
## 📊 Overall Results — {len(tests)} tests

| Métrique | Score | Status |
|----------|-------|--------|
| **Mean Reciprocal Rank (MRR)** | **{mean_mrr:.4f}** | {color_score(mean_mrr)} |
| **Normalized DCG (nDCG)** | **{mean_ndcg:.4f}** | {color_score(mean_ndcg)} |
| **Keyword Coverage** | **{mean_coverage:.1%}** | {color_score(mean_coverage)} |

### Résultats par catégorie
{category_df.to_markdown(index=False, floatfmt=".4f")}
"""
    return stats_md, fig

In [142]:
# ─── ÉVALUATION ANSWER (LLM Judge) ───────────────────────────────────────────

def run_answer_eval(progress=gr.Progress()):
    results = []
    for i, test in enumerate(tests):
        progress((i + 1) / len(tests), desc=f"Évaluation {i+1}/{len(tests)}...")
        score = evaluate_answer(test)
        results.append({
            "category": test.category,
            "question": test.question[:50] + "...",
            "accuracy": score.accuracy,
            "completeness": score.completeness,
            "relevance": score.relevance,
            "overall": score.overall,
        })
    df = pd.DataFrame(results)
    
    mean_accuracy = df["accuracy"].mean()
    mean_completeness = df["completeness"].mean()
    mean_relevance = df["relevance"].mean()
    mean_overall = df["overall"].mean()
    
    cat_df = df.groupby("category")[["accuracy", "completeness", "relevance"]].mean().reset_index()
    cat_melted = cat_df.melt(id_vars="category", var_name="dimension", value_name="score")
    
    fig = px.bar(
        cat_melted, x="category", y="score", color="dimension", barmode="group",
        title="Answer Quality by Category", labels={"score": "Score /5", "category": "Category"},
        color_discrete_map={"accuracy": "#3B82F6", "completeness": "#10B981", "relevance": "#F59E0B"},
    )
    fig.update_layout(yaxis_range=[0, 5])
    
    def color_score(val, max_val=5):
        if (val / max_val) >= 0.8: return "🟢"
        elif (val / max_val) >= 0.6: return "🟡"
        else: return "🔴"
    
    stats_md = f"""
## 🤖 Answer Quality — LLM Judge ({len(tests)} questions)

| Dimension | Score | Status |
|-----------|-------|--------|
| **Accuracy** | **{mean_accuracy:.2f}/5** | {color_score(mean_accuracy)} |
| **Completeness**| **{mean_completeness:.2f}/5** | {color_score(mean_completeness)} |
| **Relevance** | **{mean_relevance:.2f}/5** | {color_score(mean_relevance)} |
| **Overall** | **{mean_overall:.2f}/5** | {color_score(mean_overall)} |
"""
    return stats_md, fig

In [ ]:
# ─── INTERFACE GRADIO ─────────────────────────────────────────────────────────

def build_ui():
    with gr.Blocks(title="RAG Evaluation Dashboard") as demo:
        
        gr.Markdown("# 📊 RAG Evaluation Dashboard & Vector Space")
        
        # ── TAB 1: RETRIEVAL EVALUATION ──
        with gr.Tab("🔍 Retrieval Evaluation"):
            retrieval_btn = gr.Button("▶ Run Retrieval Evaluation", variant="primary")
            with gr.Row():
                retrieval_stats = gr.Markdown("*Clique sur le bouton pour lancer l'évaluation...*")
            retrieval_plot = gr.Plot()
            retrieval_btn.click(fn=run_retrieval_eval, outputs=[retrieval_stats, retrieval_plot])
        
        # ── TAB 2: ANSWER EVALUATION ──
        with gr.Tab("🤖 Answer Evaluation"):
            answer_btn = gr.Button("▶ Run Answer Evaluation", variant="primary")
            with gr.Row():
                answer_stats = gr.Markdown("*Clique sur le bouton pour lancer l'évaluation...*")
            answer_plot = gr.Plot()
            answer_btn.click(fn=run_answer_eval, outputs=[answer_stats, answer_plot])
            
        # ── TAB 3: VISUALISATION VECTEURS (NOUVEAU) ──
        with gr.Tab("🌌 Visualisation des Vecteurs"):
            gr.Markdown("""
            ### Exploration de l'espace latent
            Cette vue réduit les dimensions de tes vecteurs (via t-SNE) pour les afficher en 2D ou 3D. 
            - **Couleurs** : Catégories de documents (Type).
            - **Points proches** : Signification sémantique similaire.
            """)
            
            with gr.Row():
                btn_2d = gr.Button("📊 Générer Vue 2D", variant="secondary")
                btn_3d = gr.Button("🧊 Générer Vue 3D", variant="primary")
            
            # Affichage dynamique du graphique
            vector_plot = gr.Plot()
            
            btn_2d.click(fn=plot_2d_vectors, outputs=[vector_plot])
            btn_3d.click(fn=plot_3d_vectors, outputs=[vector_plot])
        
        # ── TAB 4: TEST INTERACTIF ──
        with gr.Tab("💬 Test interactif"):
            question_input = gr.Textbox(label="Ta question", placeholder="Ex: Quelle est la politique...", lines=2)
            ask_btn = gr.Button("Poser la question", variant="primary")
            answer_output = gr.Textbox(label="Réponse RAG", lines=6)
            ask_btn.click(fn=answer_question, inputs=[question_input], outputs=[answer_output])
    
    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(
        server_name="0.0.0.0",
        theme=gr.themes.Soft(),
        css=".metric-box { border-radius: 8px; padding: 16px; margin: 8px; } h1 { color: #1e293b; }",
        share=False,
    )

* Running on local URL:  http://0.0.0.0:7863
* To create a public link, set `share=True` in `launch()`.


⚠️  Erreur juge LLM: 2 validation errors for EvaluationScores
completeness
  Field required [type=missing, input_value={'accuracy': 1, 'complét...ponse de référence."}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
relevance
  Field required [type=missing, input_value={'accuracy': 1, 'complét...ponse de référence."}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
⚠️  Erreur juge LLM: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knaef6qwf0nrgfetxtcaf6s4` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99656, Requested 415. Please try again in 1m1.344s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
⚠️  Erreur juge LLM: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versa

## Interface avec Streamlit

In [126]:
import streamlit as st
import pandas as pd
import plotly.express as px
import time

from evaluation.test import load_tests
from evaluation.eval import evaluate_retrieval, evaluate_answer

# ─── CONFIGURATION DE LA PAGE ───
st.set_page_config(
    page_title="RAG Evaluation Studio",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ─── CHARGEMENT EN CACHE ───
@st.cache_data
def get_test_data():
    """Mise en cache des tests pour éviter de relire le fichier JSONL en boucle"""
    try:
        return load_tests()
    except Exception as e:
        st.error(f"Erreur lors du chargement des tests : {e}")
        return []

tests = get_test_data()

# ─── SIDEBAR ───
with st.sidebar:
    st.title("⚙️ Paramètres")
    st.info(f"**{len(tests)}** questions de test chargées.")
    st.markdown("---")
    st.markdown("🛠️ **Modèles utilisés :**")
    st.markdown("- **Embedding:** `all-MiniLM-L6-v2`")
    st.markdown("- **RAG LLM:** `llama-4-scout-17b-16e-instruct`")
    st.markdown("- **Judge LLM:** `llama-3.3-70b-versatile`")

st.title("🧠 RAG Evaluation Studio")
st.markdown("Évalue les performances de ton système Retrieval-Augmented Generation en un clic.")

# ─── TABS ───
tab1, tab2, tab3 = st.tabs(["📊 Évaluation Retrieval", "⚖️ Évaluation Réponse (Judge)", "💬 Chat Interactif"])

# ─── TAB 1 : RETRIEVAL ───
with tab1:
    st.header("Évaluation de la Récupération (Retrieval)")
    st.write("Mesure la capacité du Vector Store (Chroma) à ramener les bons chunks.")
    
    if st.button("▶ Lancer l'évaluation Retrieval", type="primary"):
        progress_text = "Évaluation en cours..."
        my_bar = st.progress(0, text=progress_text)
        
        results = []
        for i, test in enumerate(tests):
            res = evaluate_retrieval(test)
            results.append({
                "Catégorie": test.category,
                "Question": test.question[:50] + "...",
                "MRR": res.mrr,
                "NDCG": res.ndcg,
                "Coverage": res.keywords_coverage,
            })
            my_bar.progress((i + 1) / len(tests), text=f"Traitement de la question {i+1}/{len(tests)}")
            
        df_retrieval = pd.DataFrame(results)
        
        # Affichage des KPIs
        col1, col2, col3 = st.columns(3)
        col1.metric("MRR Moyen", f"{df_retrieval['MRR'].mean():.2f}")
        col2.metric("nDCG Moyen", f"{df_retrieval['nDCG'].mean():.2f}")
        col3.metric("Coverage Moyen", f"{df_retrieval['Coverage'].mean():.1%}")
        
        # Graphique
        st.subheader("Performance par Catégorie (MRR)")
        df_group = df_retrieval.groupby("Catégorie", as_index=False)["MRR"].mean()
        fig = px.bar(df_group, x="Catégorie", y="MRR", color="Catégorie", text_auto='.2f')
        fig.update_layout(showlegend=False)
        st.plotly_chart(fig, use_container_width=True)
        
        with st.expander("Voir les données brutes"):
            st.dataframe(df_retrieval, use_container_width=True)

# ─── TAB 2 : ANSWER (LLM-as-a-judge) ───
with tab2:
    st.header("Évaluation de la Génération (LLM-as-a-judge)")
    st.write("Utilise un modèle plus puissant pour noter factuellement les réponses du RAG.")
    
    if st.button("▶ Lancer l'évaluation des Réponses", type="primary"):
        progress_text = "Jugement LLM en cours (peut être long)..."
        my_bar = st.progress(0, text=progress_text)
        
        ans_results = []
        for i, test in enumerate(tests):
            res = evaluate_answer(test)
            ans_results.append({
                "Catégorie": test.category,
                "Accuracy": res.accuracy,
                "Completeness": res.completeness,
                "Relevance": res.relevance,
                "Overall": res.overall
            })
            # Respecter les rate limits de l'API gratuite Groq
            time.sleep(1) 
            my_bar.progress((i + 1) / len(tests), text=f"Jugement de la réponse {i+1}/{len(tests)}")
            
        df_ans = pd.DataFrame(ans_results)
        
        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Accuracy", f"{df_ans['Accuracy'].mean():.1f}/5")
        col2.metric("Completeness", f"{df_ans['Completeness'].mean():.1f}/5")
        col3.metric("Relevance", f"{df_ans['Relevance'].mean():.1f}/5")
        col4.metric("Score Global", f"{df_ans['Overall'].mean():.1f}/5")
        
        st.subheader("Distribution des Scores Globaux")
        fig2 = px.box(df_ans, x="Catégorie", y="Overall", points="all", color="Catégorie")
        st.plotly_chart(fig2, use_container_width=True)

# ─── TAB 3 : CHAT INTERACTIF ───
with tab3:
    st.header("Chat avec le Système RAG")
    
    # Initialiser l'historique du chat dans la session Streamlit
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Afficher l'historique
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    # Input utilisateur
    if prompt := st.chat_input("Pose une question sur tes documents (ex: politique de congés)..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)

        # Générer et afficher la réponse
        with st.chat_message("assistant"):
            with st.spinner("Recherche et réflexion..."):
                try:
                    reponse_rag = answer_question(prompt)
                    st.markdown(reponse_rag)
                    st.session_state.messages.append({"role": "assistant", "content": reponse_rag})
                except Exception as e:
                    st.error(f"Erreur de génération : {e}")

2026-04-07 10:35:35.816 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 10:35:35.817 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-07 10:35:35.818 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-07 10:35:35.821 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 10:35:35.832 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 10:35:36.047 
  command:

    streamlit run /home/bedane/dev/llm_engeneering/.venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-04-07 10:35:36.04